# Nested-CV outer-test evaluation
Runs the locked outer-test evaluation after hyperparameter/model selection has been locked for all five outer folds.

### Fixed evaluation protocol

1. Locate and validate `locked_selection.json` for outer folds **0–4**.
2. Verify every fold was locked before test use and has five selected checkpoints for seeds **17, 42, 73, 101, 137**.
3. Only after all locks pass validation, load each fold's `test` role.
4. Evaluate every locked checkpoint **without training or tuning anything**.
5. Save subject-level predictions for the 3MT pathway, TMC pathway, and fixed-equal hybrid.
6. Report checkpoint-level test metrics and a pre-specified **five-seed probability ensemble** for each outer fold.
7. Concatenate the five fold ensembles into one out-of-fold (OOF) prediction table in which each ADNI participant appears exactly once.
8. Save uncertainty/conflict quantities for later error-detection analysis.

there is no Optuna, optimizer, scheduler, early stopping, or test-driven model selection in this notebook.


In [ ]:
# 1. Environment, paths, and frozen evaluation settings

from pathlib import Path
import gc
import hashlib
import json
import math
import os
import random
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    roc_curve,
)

from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore", category=UserWarning)

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False,
)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/adni_mri"
)

MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "3mt_tmc_evidential"
)

MANIFEST_DIR = (
    MODEL_ROOT
    / "manifests"
)

INPUT_DIR = (
    MODEL_ROOT
    / "final_task_ready_inputs"
    / "mci_prognosis"
)

EXPERIMENTS_ROOT = (
    MODEL_ROOT
    / "experiments"
)

# Standard location used by folds 1-4 in the supplied notebooks.
SELECTION_ROOT = (
    EXPERIMENTS_ROOT
    / "per_fold_hyperparameter_selection_fixed_equal"
)

FINAL_MODEL_SCHEMA_PATH = (
    MANIFEST_DIR
    / "final_model_input_schema.json"
)

OUTPUT_ROOT = (
    EXPERIMENTS_ROOT
    / "final_nested_cv_test_evaluation_fixed_equal"
)

SEED_PREDICTION_ROOT = (
    OUTPUT_ROOT
    / "seed_predictions"
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SEED_PREDICTION_ROOT.mkdir(parents=True, exist_ok=True)

OUTER_FOLDS = [0, 1, 2, 3, 4]
EXPECTED_SEEDS = [17, 42, 73, 101, 137]

BATCH_SIZE = 2
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

# Safe resume: completed checkpoint predictions are reused only if their
# IDs/row count still match the current fold's test table.
RESUME_COMPLETED_CHECKPOINTS = True

# Fold 0 was selected in a separate exploratory notebook. Leave this as None
# if the manifest can be found automatically. If auto-discovery finds zero or
# multiple fold-0 manifests, paste the Drive path here.
LOCKED_MANIFEST_OVERRIDES = {
    0: None,
    1: None,
    2: None,
    3: None,
    4: None,
}

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 78)
print("FINAL NESTED-CV OUTER-TEST EVALUATION")
print("=" * 78)
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Outer folds: {OUTER_FOLDS}")
print(f"Locked seeds: {EXPECTED_SEEDS}")


## 2. Load the same prepared schema used during selection

This is copied from the fold-selection notebooks so the final evaluation uses the same target, feature masks, branch masks, modality order, and MRI shape.


In [ ]:
# 2. Load the prepared schema

with open(
    FINAL_MODEL_SCHEMA_PATH,
    "r",
    encoding="utf-8",
) as handle:
    final_model_schema = json.load(handle)

identifier_columns = final_model_schema[
    "identifier_columns"
]

target_column = final_model_schema[
    "model_target_column"
]

branch_mask_columns = final_model_schema[
    "branch_mask_columns"
]

feature_mask_columns = final_model_schema[
    "feature_mask_columns"
]

scaled_continuous_columns = final_model_schema[
    "scaled_continuous_columns"
]

encoded_categorical_columns = final_model_schema[
    "encoded_categorical_columns"
]

mri_path_columns = final_model_schema[
    "mri_path_columns"
]

MODALITY_ORDER = [
    "demographics",
    "cognitive_functional",
    "csf",
    "plasma",
    "apoe",
    "mri",
]

THREE_MT_CASCADE_ORDER = [
    "demographics",
    "apoe",
    "cognitive_functional",
    "csf",
    "plasma",
    "mri",
]

NUMBER_OF_CLASSES = 2

PROGNOSIS_CLASS_ORDER = [
    "sMCI",
    "pMCI",
]

MRI_SPATIAL_SHAPE = (
    177,
    213,
    183,
)

MRI_TENSOR_SHAPE = (
    1,
    *MRI_SPATIAL_SHAPE,
)

print("=" * 72)
print("PREPARED INPUT SCHEMA")
print("=" * 72)
print(f"Target column: {target_column}")
print(f"Scaled continuous features: {len(scaled_continuous_columns)}")
print(f"Encoded categorical features: {len(encoded_categorical_columns)}")
print(f"Branch masks: {len(branch_mask_columns)}")
print(f"Feature masks: {len(feature_mask_columns)}")
print(f"MRI path columns: {len(mri_path_columns)}")
print(f"Modality order: {MODALITY_ORDER}")
print(f"3MT cascade order: {THREE_MT_CASCADE_ORDER}")


## 3. Preserve the branch-level input contract

No feature engineering, imputation, relabelling, or rescaling is repeated here.


In [ ]:
# 3. Preserve the multimodal dataset contract

dataset_column_contract = {'demographics': {'continuous': ['DEMOGRAPHICS__AGE_AT_BASELINE__Z',
                                 'DEMOGRAPHICS__PTEDUCAT_CLEAN__Z'],
                  'categorical': ['DEMOGRAPHICS__PTGENDER_CLEAN__IDX',
                                  'DEMOGRAPHICS__PTHAND_CLEAN__IDX'],
                  'feature_masks': ['FEATURE_MASK__DEMOGRAPHICS_AGE_AT_BASELINE',
                                    'FEATURE_MASK__DEMOGRAPHICS_PTEDUCAT_CLEAN',
                                    'FEATURE_MASK__DEMOGRAPHICS_PTGENDER_CLEAN',
                                    'FEATURE_MASK__DEMOGRAPHICS_PTHAND_CLEAN'],
                  'branch_mask': 'BRANCH_MASK__DEMOGRAPHICS'},
 'cognitive_functional': {'continuous': ['ADAS__TOTSCORE__Z',
                                         'ADAS__TOTAL13__Z',
                                         'MMSE__MMSE_TOTAL_SCORE__Z',
                                         'MMSE__MMSE_ORIENTATION_SCORE__Z',
                                         'MMSE__MMSE_REGISTRATION_SCORE__Z',
                                         'MMSE__MMSE_ATTENTION_SCORE__Z',
                                         'MMSE__MMSE_DELAYED_RECALL_SCORE__Z',
                                         'MMSE__MMSE_LANGUAGE_COMMAND_SCORE__Z',
                                         'FAQ__FAQTOTAL__Z'],
                          'categorical': [],
                          'feature_masks': ['FEATURE_MASK__ADAS_TOTSCORE',
                                            'FEATURE_MASK__ADAS_TOTAL13',
                                            'FEATURE_MASK__MMSE_MMSE_TOTAL_SCORE',
                                            'FEATURE_MASK__MMSE_MMSE_ORIENTATION_SCORE',
                                            'FEATURE_MASK__MMSE_MMSE_REGISTRATION_SCORE',
                                            'FEATURE_MASK__MMSE_MMSE_ATTENTION_SCORE',
                                            'FEATURE_MASK__MMSE_MMSE_DELAYED_RECALL_SCORE',
                                            'FEATURE_MASK__MMSE_MMSE_LANGUAGE_COMMAND_SCORE',
                                            'FEATURE_MASK__FAQ_FAQTOTAL'],
                          'branch_mask': 'BRANCH_MASK__COGNITIVE_FUNCTIONAL'},
 'csf': {'continuous': ['CSF__ABETA40__Z',
                        'CSF__ABETA42__Z',
                        'CSF__TAU__Z',
                        'CSF__PTAU__Z',
                        'CSF__ABETA42_40_RATIO__Z'],
         'categorical': [],
         'feature_masks': ['FEATURE_MASK__CSF_ABETA40',
                           'FEATURE_MASK__CSF_ABETA42',
                           'FEATURE_MASK__CSF_TAU',
                           'FEATURE_MASK__CSF_PTAU',
                           'FEATURE_MASK__CSF_ABETA42_40_RATIO'],
         'branch_mask': 'BRANCH_MASK__CSF'},
 'plasma': {'continuous': ['PLASMA__pT217_F__Z',
                           'PLASMA__AB42_F__Z',
                           'PLASMA__AB40_F__Z',
                           'PLASMA__AB42_AB40_F__Z',
                           'PLASMA__pT217_AB42_F__Z',
                           'PLASMA__NfL_Q__Z',
                           'PLASMA__GFAP_Q__Z',
                           'PLASMA__NfL_F__Z',
                           'PLASMA__GFAP_F__Z'],
            'categorical': [],
            'feature_masks': ['FEATURE_MASK__PLASMA_pT217_F',
                              'FEATURE_MASK__PLASMA_AB42_F',
                              'FEATURE_MASK__PLASMA_AB40_F',
                              'FEATURE_MASK__PLASMA_AB42_AB40_F',
                              'FEATURE_MASK__PLASMA_pT217_AB42_F',
                              'FEATURE_MASK__PLASMA_NfL_Q',
                              'FEATURE_MASK__PLASMA_GFAP_Q',
                              'FEATURE_MASK__PLASMA_NfL_F',
                              'FEATURE_MASK__PLASMA_GFAP_F'],
            'branch_mask': 'BRANCH_MASK__PLASMA'},
 'apoe': {'continuous': [],
          'categorical': ['APOE__APOE4_ALLELE_COUNT__IDX'],
          'feature_masks': ['FEATURE_MASK__APOE_APOE4_ALLELE_COUNT'],
          'branch_mask': 'BRANCH_MASK__APOE'},
 'mri': {'path': 'MRI__NORMALIZED_T1_NPY_PATH', 'branch_mask': 'BRANCH_MASK__MRI'}}

print("=" * 72)
print("DATASET CONTRACT")
print("=" * 72)

for branch_name, definition in dataset_column_contract.items():
    print(
        f"{branch_name:24s} | "
        f"continuous={len(definition.get('continuous', [])):2d} | "
        f"categorical={len(definition.get('categorical', [])):2d} | "
        f"feature masks={len(definition.get('feature_masks', [])):2d}"
    )


## 4. Dataset class

This is the same multimodal dataset implementation used in the selection notebooks. A test DataLoader is **not** created yet.


In [ ]:
# 4. Multimodal dataset and train/validation DataLoader factory

class ADNIMultimodalDataset(Dataset):
    """
    Dataset for the already-prepared ADNI multimodal tables.

    No preprocessing, relabelling, rescaling, or imputation is
    repeated here. MRI arrays are loaded lazily from the prepared
    NumPy paths.
    """

    def __init__(
        self,
        dataframe,
        column_contract,
        branch_mask_order,
        feature_mask_order,
        target_column,
        load_mri=True,
    ):
        self.dataframe = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.column_contract = column_contract
        self.branch_mask_order = list(branch_mask_order)
        self.feature_mask_order = list(feature_mask_order)
        self.target_column = target_column
        self.load_mri = load_mri

    def __len__(self):
        return len(self.dataframe)

    @staticmethod
    def _continuous_tensor(row, columns):
        if not columns:
            return None

        values = row[columns].to_numpy(
            dtype=np.float32
        )

        return torch.from_numpy(values)

    @staticmethod
    def _categorical_tensor(row, columns):
        if not columns:
            return None

        values = row[columns].to_numpy(
            dtype=np.int64
        )

        return torch.from_numpy(values)

    @staticmethod
    def _mask_tensor(row, columns):
        if not columns:
            return None

        values = row[columns].to_numpy(
            dtype=np.float32
        )

        return torch.from_numpy(values)

    def _load_mri_tensor(
        self,
        mri_path,
        mri_branch_mask,
    ):
        if float(mri_branch_mask) == 0.0:
            return torch.zeros(
                MRI_TENSOR_SHAPE,
                dtype=torch.float32,
            )

        if not self.load_mri:
            return torch.zeros(
                MRI_TENSOR_SHAPE,
                dtype=torch.float32,
            )

        if pd.isna(mri_path):
            raise ValueError(
                "MRI branch mask is 1 but the prepared MRI path is missing."
            )

        mri_path = Path(
            str(mri_path)
        )

        if not mri_path.exists():
            raise FileNotFoundError(
                f"Prepared MRI array not found: {mri_path}"
            )

        mri_array = np.load(
            mri_path,
            allow_pickle=False,
        )

        if tuple(mri_array.shape) != MRI_SPATIAL_SHAPE:
            raise ValueError(
                f"Unexpected MRI shape {mri_array.shape} at {mri_path}"
            )

        mri_array = np.asarray(
            mri_array,
            dtype=np.float32,
        )

        mri_array = np.expand_dims(
            mri_array,
            axis=0,
        )

        return torch.from_numpy(
            mri_array
        )

    def __getitem__(self, index):
        row = self.dataframe.iloc[
            index
        ]

        modalities = {}

        demographics = self.column_contract[
            "demographics"
        ]

        modalities["demographics"] = {
            "continuous":
                self._continuous_tensor(
                    row,
                    demographics["continuous"],
                ),
            "categorical":
                self._categorical_tensor(
                    row,
                    demographics["categorical"],
                ),
            "feature_mask":
                self._mask_tensor(
                    row,
                    demographics["feature_masks"],
                ),
            "branch_mask":
                torch.tensor(
                    row[
                        demographics["branch_mask"]
                    ],
                    dtype=torch.float32,
                ),
        }

        for branch_name in [
            "cognitive_functional",
            "csf",
            "plasma",
        ]:
            definition = self.column_contract[
                branch_name
            ]

            modalities[
                branch_name
            ] = {
                "continuous":
                    self._continuous_tensor(
                        row,
                        definition["continuous"],
                    ),
                "feature_mask":
                    self._mask_tensor(
                        row,
                        definition["feature_masks"],
                    ),
                "branch_mask":
                    torch.tensor(
                        row[
                            definition["branch_mask"]
                        ],
                        dtype=torch.float32,
                    ),
            }

        apoe = self.column_contract[
            "apoe"
        ]

        modalities["apoe"] = {
            "categorical":
                self._categorical_tensor(
                    row,
                    apoe["categorical"],
                ),
            "feature_mask":
                self._mask_tensor(
                    row,
                    apoe["feature_masks"],
                ),
            "branch_mask":
                torch.tensor(
                    row[
                        apoe["branch_mask"]
                    ],
                    dtype=torch.float32,
                ),
        }

        mri = self.column_contract[
            "mri"
        ]

        mri_branch_mask = row[
            mri["branch_mask"]
        ]

        modalities["mri"] = {
            "image":
                self._load_mri_tensor(
                    mri_path=row[
                        mri["path"]
                    ],
                    mri_branch_mask=mri_branch_mask,
                ),
            "branch_mask":
                torch.tensor(
                    mri_branch_mask,
                    dtype=torch.float32,
                ),
        }

        return {
            "rid":
                int(row["RID"]),
            "ptid":
                str(row["PTID"]),
            "target":
                torch.tensor(
                    int(
                        row[
                            self.target_column
                        ]
                    ),
                    dtype=torch.long,
                ),
            "modalities":
                modalities,
            "branch_masks":
                self._mask_tensor(
                    row,
                    self.branch_mask_order,
                ),
            "feature_masks":
                self._mask_tensor(
                    row,
                    self.feature_mask_order,
                ),
        }


## 5. Model architecture

The following architecture code is copied unchanged from the supplied per-fold selection notebooks.


In [ ]:
# 5. Searchable version of the completed architecture

def _scaled_width(
    base_width,
    scale,
):
    """
    Scale the existing hidden width while keeping widths compact
    and aligned to multiples of eight.
    """
    scaled = int(
        round(
            base_width
            * float(scale)
            / 8.0
        )
        * 8
    )

    return max(
        8,
        scaled,
    )

class MaskAwareScalarEncoder(nn.Module):
    def __init__(
        self,
        value_dim,
        mask_dim,
        output_dim,
        hidden_dim,
        dropout,
    ):
        super().__init__()

        input_dim = (
            value_dim
            + mask_dim
        )

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(
                hidden_dim
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim,
                output_dim,
            ),
            nn.LayerNorm(
                output_dim
            ),
            nn.GELU(),
        )

    def forward(
        self,
        values,
        feature_mask,
    ):
        inputs = torch.cat(
            [
                values,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(
            inputs
        )

class DemographicsEncoder(nn.Module):
    def __init__(
        self,
        output_dim,
        hidden_dim,
        dropout,
    ):
        super().__init__()

        self.sex_embedding = nn.Embedding(
            num_embeddings=3,
            embedding_dim=4,
            padding_idx=0,
        )

        self.handedness_embedding = nn.Embedding(
            num_embeddings=3,
            embedding_dim=4,
            padding_idx=0,
        )

        input_dim = (
            2
            + 4
            + 4
            + 4
        )

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(
                hidden_dim
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim,
                output_dim,
            ),
            nn.LayerNorm(
                output_dim
            ),
            nn.GELU(),
        )

    def forward(
        self,
        continuous,
        categorical,
        feature_mask,
    ):
        sex_representation = (
            self.sex_embedding(
                categorical[
                    :,
                    0,
                ]
            )
        )

        handedness_representation = (
            self.handedness_embedding(
                categorical[
                    :,
                    1,
                ]
            )
        )

        inputs = torch.cat(
            [
                continuous,
                sex_representation,
                handedness_representation,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(
            inputs
        )

class APOEEncoder(nn.Module):
    def __init__(
        self,
        output_dim,
        hidden_dim,
        dropout,
    ):
        super().__init__()

        self.apoe_embedding = nn.Embedding(
            num_embeddings=4,
            embedding_dim=8,
            padding_idx=0,
        )

        self.network = nn.Sequential(
            nn.Linear(
                9,
                hidden_dim,
            ),
            nn.LayerNorm(
                hidden_dim
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim,
                output_dim,
            ),
            nn.LayerNorm(
                output_dim
            ),
            nn.GELU(),
        )

    def forward(
        self,
        categorical,
        feature_mask,
    ):
        apoe_representation = (
            self.apoe_embedding(
                categorical[
                    :,
                    0,
                ]
            )
        )

        inputs = torch.cat(
            [
                apoe_representation,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(
            inputs
        )

class ResidualDownsampleBlock3D(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        self.pool = nn.MaxPool3d(
            kernel_size=2,
            stride=2,
        )

        self.main_path = nn.Sequential(
            nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
            nn.GELU(),
            nn.Conv3d(
                out_channels,
                out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
        )

        self.residual_path = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=1,
            stride=1,
            bias=False,
        )

        self.activation = nn.GELU()

    def forward(
        self,
        inputs,
    ):
        pooled = self.pool(
            inputs
        )

        return self.activation(
            self.main_path(
                pooled
            )
            +
            self.residual_path(
                pooled
            )
        )

class MRIEncoder3D(nn.Module):
    """
    The MRI CNN topology, 256-dimensional patch representation,
    one transformer layer, eight MRI attention heads, and 512-unit
    MRI feed-forward layer remain fixed.

    Only the shared projection dimension and dropout scale are tuned.
    """

    def __init__(
        self,
        output_dim,
        input_shape=MRI_SPATIAL_SHAPE,
        dropout=0.20,
    ):
        super().__init__()

        patch_embedding_dim = 256
        transformer_heads = 8
        transformer_layers = 1
        transformer_feedforward_dim = 512

        self.input_shape = tuple(
            input_shape
        )

        self.stem = nn.Sequential(
            nn.Conv3d(
                1,
                16,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),
            nn.Conv3d(
                16,
                16,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),
        )

        self.residual_blocks = nn.Sequential(
            ResidualDownsampleBlock3D(
                16,
                32,
            ),
            ResidualDownsampleBlock3D(
                32,
                64,
            ),
            ResidualDownsampleBlock3D(
                64,
                128,
            ),
            ResidualDownsampleBlock3D(
                128,
                256,
            ),
        )

        stem_shape = tuple(
            math.ceil(
                dimension
                / 2
            )
            for dimension
            in self.input_shape
        )

        patch_grid_shape = (
            stem_shape
        )

        for _ in range(4):
            patch_grid_shape = tuple(
                dimension
                // 2
                for dimension
                in patch_grid_shape
            )

        self.number_of_patches = math.prod(
            patch_grid_shape
        )

        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                self.number_of_patches,
                patch_embedding_dim,
            )
        )

        nn.init.trunc_normal_(
            self.position_embedding,
            std=0.02,
        )

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=patch_embedding_dim,
            nhead=transformer_heads,
            dim_feedforward=transformer_feedforward_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer=transformer_layer,
            num_layers=transformer_layers,
            norm=nn.LayerNorm(
                patch_embedding_dim
            ),
        )

        self.projection = nn.Sequential(
            nn.Linear(
                patch_embedding_dim,
                output_dim,
            ),
            nn.LayerNorm(
                output_dim
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
        )

    def forward(
        self,
        image,
    ):
        feature_map = self.stem(
            image
        )

        feature_map = self.residual_blocks(
            feature_map
        )

        batch_size, channels, depth, height, width = (
            feature_map.shape
        )

        actual_patch_count = (
            depth
            * height
            * width
        )

        if actual_patch_count != self.number_of_patches:
            raise ValueError(
                "Unexpected MRI patch count. "
                f"Expected {self.number_of_patches}, "
                f"obtained {actual_patch_count}."
            )

        patch_tokens = (
            feature_map
            .flatten(
                start_dim=2
            )
            .transpose(
                1,
                2,
            )
        )

        patch_tokens = (
            patch_tokens
            + self.position_embedding
        )

        transformed_tokens = (
            self.transformer_encoder(
                patch_tokens
            )
        )

        pooled = transformed_tokens.mean(
            dim=1
        )

        return self.projection(
            pooled
        )

class ADNIModalityEncoders(nn.Module):
    """
    Searchable wrapper around the same six branch encoders.

    At width_scale=1.0 and dropout_scale=1.0 the hidden widths and
    dropout values match the completed fixed-fusion implementation.
    """

    def __init__(
        self,
        output_dim,
        tabular_width_scale,
        dropout_scale,
    ):
        super().__init__()

        scalar_dropout = min(
            0.50,
            0.20
            * dropout_scale,
        )

        apoe_dropout = min(
            0.50,
            0.10
            * dropout_scale,
        )

        mri_dropout = min(
            0.50,
            0.20
            * dropout_scale,
        )

        self.demographics = DemographicsEncoder(
            output_dim=output_dim,
            hidden_dim=_scaled_width(
                48,
                tabular_width_scale,
            ),
            dropout=scalar_dropout,
        )

        self.cognitive_functional = MaskAwareScalarEncoder(
            value_dim=9,
            mask_dim=9,
            output_dim=output_dim,
            hidden_dim=_scaled_width(
                96,
                tabular_width_scale,
            ),
            dropout=scalar_dropout,
        )

        self.csf = MaskAwareScalarEncoder(
            value_dim=5,
            mask_dim=5,
            output_dim=output_dim,
            hidden_dim=_scaled_width(
                64,
                tabular_width_scale,
            ),
            dropout=scalar_dropout,
        )

        self.plasma = MaskAwareScalarEncoder(
            value_dim=9,
            mask_dim=9,
            output_dim=output_dim,
            hidden_dim=_scaled_width(
                96,
                tabular_width_scale,
            ),
            dropout=scalar_dropout,
        )

        self.apoe = APOEEncoder(
            output_dim=output_dim,
            hidden_dim=_scaled_width(
                32,
                tabular_width_scale,
            ),
            dropout=apoe_dropout,
        )

        self.mri = MRIEncoder3D(
            output_dim=output_dim,
            input_shape=MRI_SPATIAL_SHAPE,
            dropout=mri_dropout,
        )

    def forward(
        self,
        modalities,
    ):
        return {
            "demographics":
                self.demographics(
                    continuous=
                        modalities[
                            "demographics"
                        ]["continuous"],
                    categorical=
                        modalities[
                            "demographics"
                        ]["categorical"],
                    feature_mask=
                        modalities[
                            "demographics"
                        ]["feature_mask"],
                ),
            "cognitive_functional":
                self.cognitive_functional(
                    values=
                        modalities[
                            "cognitive_functional"
                        ]["continuous"],
                    feature_mask=
                        modalities[
                            "cognitive_functional"
                        ]["feature_mask"],
                ),
            "csf":
                self.csf(
                    values=
                        modalities[
                            "csf"
                        ]["continuous"],
                    feature_mask=
                        modalities[
                            "csf"
                        ]["feature_mask"],
                ),
            "plasma":
                self.plasma(
                    values=
                        modalities[
                            "plasma"
                        ]["continuous"],
                    feature_mask=
                        modalities[
                            "plasma"
                        ]["feature_mask"],
                ),
            "apoe":
                self.apoe(
                    categorical=
                        modalities[
                            "apoe"
                        ]["categorical"],
                    feature_mask=
                        modalities[
                            "apoe"
                        ]["feature_mask"],
                ),
            "mri":
                self.mri(
                    modalities[
                        "mri"
                    ]["image"]
                ),
        }

class MaskedADNIModalityEncoders(nn.Module):
    def __init__(
        self,
        output_dim,
        tabular_width_scale,
        dropout_scale,
    ):
        super().__init__()

        self.encoders = ADNIModalityEncoders(
            output_dim=output_dim,
            tabular_width_scale=
                tabular_width_scale,
            dropout_scale=
                dropout_scale,
        )

    @staticmethod
    def _apply_branch_mask(
        representation,
        branch_mask,
    ):
        return (
            representation
            * branch_mask.unsqueeze(
                -1
            )
        )

    def forward(
        self,
        modalities,
    ):
        raw = self.encoders(
            modalities
        )

        masked = {}

        for modality_name in MODALITY_ORDER:
            masked[
                modality_name
            ] = self._apply_branch_mask(
                representation=
                    raw[
                        modality_name
                    ],
                branch_mask=
                    modalities[
                        modality_name
                    ]["branch_mask"],
            )

        return {
            "raw":
                raw,
            "masked":
                masked,
        }

class CascadedModalityTransformer(nn.Module):
    def __init__(
        self,
        embedding_dim,
        number_of_heads,
        dropout,
    ):
        super().__init__()

        if (
            embedding_dim
            % number_of_heads
            != 0
        ):
            raise ValueError(
                "embedding_dim must be divisible by number_of_heads"
            )

        self.self_attention_norm = nn.LayerNorm(
            embedding_dim
        )

        self.self_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=number_of_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.self_attention_dropout = nn.Dropout(
            dropout
        )

        self.cross_attention_norm = nn.LayerNorm(
            embedding_dim
        )

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=number_of_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.cross_attention_dropout = nn.Dropout(
            dropout
        )

        self.output_norm = nn.LayerNorm(
            embedding_dim
        )

    def forward(
        self,
        latent_query,
        modality_embedding,
    ):
        normalised_query = (
            self.self_attention_norm(
                latent_query
            )
        )

        self_attention_output, self_attention_weights = (
            self.self_attention(
                query=normalised_query,
                key=normalised_query,
                value=normalised_query,
                need_weights=True,
                average_attn_weights=False,
            )
        )

        self_attended_query = (
            latent_query
            + self.self_attention_dropout(
                self_attention_output
            )
        )

        normalised_self_query = (
            self.cross_attention_norm(
                self_attended_query
            )
        )

        cross_attention_output, cross_attention_weights = (
            self.cross_attention(
                query=normalised_self_query,
                key=modality_embedding,
                value=modality_embedding,
                need_weights=True,
                average_attn_weights=False,
            )
        )

        updated_query = (
            self_attended_query
            + self.cross_attention_dropout(
                cross_attention_output
            )
        )

        updated_query = self.output_norm(
            updated_query
        )

        return {
            "updated_query":
                updated_query,
            "self_attention_weights":
                self_attention_weights,
            "cross_attention_weights":
                cross_attention_weights,
        }

class ThreeMTCascade(nn.Module):
    """
    Availability-gated six-stage 3MT cascade.

    If a branch is absent, the previous query is copied exactly.
    """

    def __init__(
        self,
        embedding_dim,
        modality_order,
        cascade_order,
        number_of_heads,
        dropout,
    ):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.modality_order = list(
            modality_order
        )
        self.cascade_order = list(
            cascade_order
        )

        self.modality_to_mask_index = {
            modality_name:
                modality_index
            for modality_index, modality_name
            in enumerate(
                self.modality_order
            )
        }

        self.learned_latent_query = nn.Parameter(
            torch.empty(
                1,
                1,
                embedding_dim,
            )
        )

        nn.init.normal_(
            self.learned_latent_query,
            mean=0.0,
            std=0.02,
        )

        self.cmt_blocks = nn.ModuleDict(
            {
                modality_name:
                    CascadedModalityTransformer(
                        embedding_dim=
                            embedding_dim,
                        number_of_heads=
                            number_of_heads,
                        dropout=
                            dropout,
                    )
                for modality_name
                in self.cascade_order
            }
        )

    def forward(
        self,
        masked_representations,
        branch_masks,
    ):
        first_modality = (
            self.cascade_order[
                0
            ]
        )

        batch_size = (
            masked_representations[
                first_modality
            ].shape[
                0
            ]
        )

        latent_query = (
            self.learned_latent_query
            .expand(
                batch_size,
                -1,
                -1,
            )
        )

        stage_queries = {}
        self_attention_weights = {}
        cross_attention_weights = {}

        for modality_name in self.cascade_order:
            previous_query = (
                latent_query
            )

            modality_token = (
                masked_representations[
                    modality_name
                ]
                .unsqueeze(
                    1
                )
            )

            stage_output = (
                self.cmt_blocks[
                    modality_name
                ](
                    latent_query=
                        previous_query,
                    modality_embedding=
                        modality_token,
                )
            )

            candidate_query = (
                stage_output[
                    "updated_query"
                ]
            )

            modality_index = (
                self.modality_to_mask_index[
                    modality_name
                ]
            )

            availability = (
                branch_masks[
                    :,
                    modality_index,
                ]
                .view(
                    -1,
                    1,
                    1,
                )
                .to(
                    dtype=
                        previous_query.dtype
                )
            )

            latent_query = (
                availability
                * candidate_query
                +
                (
                    1.0
                    - availability
                )
                * previous_query
            )

            stage_queries[
                modality_name
            ] = latent_query

            self_attention_weights[
                modality_name
            ] = stage_output[
                "self_attention_weights"
            ]

            cross_attention_weights[
                modality_name
            ] = stage_output[
                "cross_attention_weights"
            ]

        return {
            "joint_representation":
                latent_query.squeeze(
                    1
                ),
            "stage_queries":
                stage_queries,
            "self_attention_weights":
                self_attention_weights,
            "cross_attention_weights":
                cross_attention_weights,
        }

class EvidentialClassificationHead(nn.Module):
    def __init__(
        self,
        input_dim,
        number_of_classes,
        hidden_dim,
        dropout,
    ):
        super().__init__()

        self.number_of_classes = (
            number_of_classes
        )

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(
                hidden_dim
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim,
                number_of_classes,
            ),
        )

    def forward(
        self,
        representation,
        branch_mask,
    ):
        raw_evidence = F.softplus(
            self.network(
                representation
            )
        )

        effective_evidence = (
            raw_evidence
            * branch_mask.unsqueeze(
                -1
            )
        )

        alpha = (
            effective_evidence
            + 1.0
        )

        strength = alpha.sum(
            dim=-1,
            keepdim=True,
        )

        return {
            "raw_evidence":
                raw_evidence,
            "evidence":
                effective_evidence,
            "alpha":
                alpha,
            "strength":
                strength,
            "belief":
                effective_evidence
                / strength,
            "probabilities":
                alpha
                / strength,
            "uncertainty":
                self.number_of_classes
                / strength,
        }

class IndependentModalityEvidentialHeads(nn.Module):
    def __init__(
        self,
        modality_order,
        input_dim,
        number_of_classes,
        hidden_dim,
        dropout,
    ):
        super().__init__()

        self.modality_order = list(
            modality_order
        )

        self.heads = nn.ModuleDict(
            {
                modality_name:
                    EvidentialClassificationHead(
                        input_dim=
                            input_dim,
                        number_of_classes=
                            number_of_classes,
                        hidden_dim=
                            hidden_dim,
                        dropout=
                            dropout,
                    )
                for modality_name
                in self.modality_order
            }
        )

    def forward(
        self,
        modality_representations,
        modalities,
    ):
        opinions = {}

        for modality_name in self.modality_order:
            opinions[
                modality_name
            ] = self.heads[
                modality_name
            ](
                representation=
                    modality_representations[
                        modality_name
                    ],
                branch_mask=
                    modalities[
                        modality_name
                    ]["branch_mask"],
            )

        return opinions

class ThreeMTAuxiliaryClassifier(nn.Module):
    def __init__(
        self,
        input_dim,
        number_of_classes,
        hidden_dim,
        dropout,
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LeakyReLU(
                negative_slope=0.01,
            ),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim,
                number_of_classes,
            ),
        )

    def forward(
        self,
        representation,
    ):
        return self.network(
            representation
        )

class ThreeMTPredictionHeads(nn.Module):
    def __init__(
        self,
        cascade_order,
        embedding_dim,
        number_of_classes,
        evidential_hidden_dim,
        dropout,
    ):
        super().__init__()

        self.cascade_order = list(
            cascade_order
        )

        self.auxiliary_stages = (
            self.cascade_order[
                :-1
            ]
        )

        self.auxiliary_heads = nn.ModuleDict(
            {
                stage_name:
                    ThreeMTAuxiliaryClassifier(
                        input_dim=
                            embedding_dim,
                        number_of_classes=
                            number_of_classes,
                        hidden_dim=
                            embedding_dim,
                        dropout=
                            dropout,
                    )
                for stage_name
                in self.auxiliary_stages
            }
        )

        self.joint_evidential_head = (
            EvidentialClassificationHead(
                input_dim=
                    embedding_dim,
                number_of_classes=
                    number_of_classes,
                hidden_dim=
                    evidential_hidden_dim,
                dropout=
                    dropout,
            )
        )

    def forward(
        self,
        three_mt_output,
    ):
        auxiliary_logits = {}

        for stage_name in self.auxiliary_stages:
            stage_representation = (
                three_mt_output[
                    "stage_queries"
                ][
                    stage_name
                ]
                .squeeze(
                    1
                )
            )

            auxiliary_logits[
                stage_name
            ] = (
                self.auxiliary_heads[
                    stage_name
                ](
                    stage_representation
                )
            )

        joint_representation = (
            three_mt_output[
                "joint_representation"
            ]
        )

        joint_presence_mask = torch.ones(
            joint_representation.shape[
                0
            ],
            dtype=
                joint_representation.dtype,
            device=
                joint_representation.device,
        )

        joint_opinion = (
            self.joint_evidential_head(
                representation=
                    joint_representation,
                branch_mask=
                    joint_presence_mask,
            )
        )

        return {
            "auxiliary_logits":
                auxiliary_logits,
            "joint_opinion":
                joint_opinion,
        }


## 6. TMC fusion and fixed-equal hybrid fusion

The fixed fusion rule remains unchanged. During inference, `model.eval()` disables modality dropout.


In [ ]:
# 6. Unchanged TMC and fixed-equal fusion modules

class TMCFusion(nn.Module):
    """
    Fuse independent Dirichlet modality opinions using the
    reduced Dempster-Shafer combination rule used by TMC.
    """

    def __init__(
        self,
        number_of_classes,
        modality_order,
        numerical_epsilon=1e-8,
    ):
        super().__init__()

        self.number_of_classes = (
            number_of_classes
        )

        self.modality_order = list(
            modality_order
        )

        self.numerical_epsilon = (
            numerical_epsilon
        )

    def _dirichlet_to_opinion(
        self,
        alpha,
    ):
        """
        Convert Dirichlet parameters into belief masses and
        one uncertainty mass.
        """

        strength = alpha.sum(
            dim=-1,
            keepdim=True,
        )

        evidence = alpha - 1.0

        belief = evidence / strength

        uncertainty = (
            self.number_of_classes
            / strength
        )

        return {
            "evidence": evidence,
            "strength": strength,
            "belief": belief,
            "uncertainty": uncertainty,
        }

    def _combine_two(
        self,
        alpha_a,
        alpha_b,
    ):
        """
        Combine two batches of Dirichlet opinions.

        Both inputs have shape:
        (batch_size, number_of_classes).
        """

        opinion_a = self._dirichlet_to_opinion(
            alpha_a
        )

        opinion_b = self._dirichlet_to_opinion(
            alpha_b
        )

        belief_a = opinion_a["belief"]
        belief_b = opinion_b["belief"]

        uncertainty_a = opinion_a[
            "uncertainty"
        ]

        uncertainty_b = opinion_b[
            "uncertainty"
        ]

        # Conflict mass

        # The outer product contains every pairwise combination
        # between class beliefs from the two opinions.
        belief_outer_product = (
            belief_a.unsqueeze(-1)
            * belief_b.unsqueeze(-2)
        )

        total_belief_product = (
            belief_outer_product.sum(
                dim=(-2, -1)
            )
        )

        same_class_agreement = (
            torch.diagonal(
                belief_outer_product,
                dim1=-2,
                dim2=-1,
            )
            .sum(dim=-1)
        )

        # Conflict contains products assigned to different classes.
        conflict = (
            total_belief_product
            - same_class_agreement
        )

        normalisation = (
            1.0
            - conflict
        ).clamp_min(
            self.numerical_epsilon
        ).unsqueeze(-1)

        # Fused belief and uncertainty masses

        fused_belief = (
            belief_a * belief_b
            + belief_a * uncertainty_b
            + belief_b * uncertainty_a
        ) / normalisation

        fused_uncertainty = (
            uncertainty_a
            * uncertainty_b
        ) / normalisation

        # Recover the fused Dirichlet opinion

        fused_strength = (
            self.number_of_classes
            / fused_uncertainty.clamp_min(
                self.numerical_epsilon
            )
        )

        fused_evidence = (
            fused_belief
            * fused_strength
        )

        fused_alpha = (
            fused_evidence
            + 1.0
        )

        fused_probabilities = (
            fused_alpha
            / fused_alpha.sum(
                dim=-1,
                keepdim=True,
            )
        )

        return {
            "alpha": fused_alpha,
            "evidence": fused_evidence,
            "belief": fused_belief,
            "uncertainty": fused_uncertainty,
            "strength": fused_strength,
            "probabilities": fused_probabilities,
            "conflict": conflict.unsqueeze(-1),
        }

    def forward(
        self,
        modality_opinions,
    ):
        """
        Sequentially combine the modality-specific opinions in
        the fixed modality order.
        """

        first_modality = self.modality_order[0]

        fused_alpha = modality_opinions[
            first_modality
        ]["alpha"]

        fusion_history = {}

        # Retain the starting opinion so that the complete fusion
        # sequence can later be inspected.
        first_opinion = self._dirichlet_to_opinion(
            fused_alpha
        )

        fusion_history[first_modality] = {
            "alpha": fused_alpha,
            "belief": first_opinion["belief"],
            "uncertainty": first_opinion[
                "uncertainty"
            ],
            "conflict": torch.zeros(
                fused_alpha.shape[0],
                1,
                dtype=fused_alpha.dtype,
                device=fused_alpha.device,
            ),
        }

        for modality_name in self.modality_order[1:]:

            next_alpha = modality_opinions[
                modality_name
            ]["alpha"]

            combined = self._combine_two(
                alpha_a=fused_alpha,
                alpha_b=next_alpha,
            )

            fused_alpha = combined["alpha"]

            fusion_history[modality_name] = {
                "alpha": combined["alpha"],
                "belief": combined["belief"],
                "uncertainty": combined[
                    "uncertainty"
                ],
                "conflict": combined["conflict"],
            }

        # Final fused opinion

        final_strength = fused_alpha.sum(
            dim=-1,
            keepdim=True,
        )

        final_evidence = (
            fused_alpha
            - 1.0
        )

        final_belief = (
            final_evidence
            / final_strength
        )

        final_uncertainty = (
            self.number_of_classes
            / final_strength
        )

        final_probabilities = (
            fused_alpha
            / final_strength
        )

        return {
            "alpha": fused_alpha,
            "evidence": final_evidence,
            "belief": final_belief,
            "strength": final_strength,
            "uncertainty": final_uncertainty,
            "probabilities": final_probabilities,
            "fusion_history": fusion_history,
        }

def inverse_softplus(value):
    """
    Return an unconstrained value whose Softplus transformation
    is approximately equal to the requested positive value.
    """

    value_tensor = torch.as_tensor(
        value,
        dtype=torch.float32,
    )

    return torch.log(
        torch.expm1(
            value_tensor
        )
    )

class FixedEqualHybridFusion(nn.Module):
    """
    Combine calibrated 3MT and TMC evidence with fixed weights.

    The participant-specific reliability gate is removed:

        w_3MT = 0.5
        w_TMC = 0.5

    The two positive pathway evidence scales remain trainable.
    This isolates the contribution of the learned gate without
    changing the remaining fusion architecture.
    """

    def __init__(
        self,
        number_of_classes,
        number_of_modalities,
        initial_three_mt_scale=1.0,
        initial_tmc_scale=1.0,
    ):
        super().__init__()

        self.number_of_classes = number_of_classes
        self.number_of_modalities = number_of_modalities

        self.three_mt_scale_parameter = nn.Parameter(
            inverse_softplus(
                initial_three_mt_scale
            ).clone()
        )

        self.tmc_scale_parameter = nn.Parameter(
            inverse_softplus(
                initial_tmc_scale
            ).clone()
        )

    def _calculate_mean_available_conflict(
        self,
        tmc_output,
        branch_masks,
        modality_order,
    ):
        """
        Calculate the mean TMC conflict across available fusion stages.
        """

        stage_conflicts = []
        stage_masks = []

        for modality_index, modality_name in enumerate(
            modality_order[1:],
            start=1,
        ):
            stage_conflicts.append(
                tmc_output[
                    "fusion_history"
                ][modality_name]["conflict"]
            )

            stage_masks.append(
                branch_masks[
                    :,
                    modality_index,
                ].unsqueeze(-1)
            )

        stacked_conflicts = torch.stack(
            stage_conflicts,
            dim=1,
        )

        stacked_masks = torch.stack(
            stage_masks,
            dim=1,
        )

        conflict_sum = (
            stacked_conflicts
            * stacked_masks
        ).sum(
            dim=1
        )

        available_fusion_count = (
            stacked_masks.sum(
                dim=1
            ).clamp_min(1.0)
        )

        return (
            conflict_sum
            / available_fusion_count
        )

    def forward(
        self,
        joint_opinion,
        tmc_output,
        branch_masks,
        modality_order,
    ):
        three_mt_evidence = joint_opinion[
            "evidence"
        ]

        tmc_evidence = tmc_output[
            "evidence"
        ]

        three_mt_scale = F.softplus(
            self.three_mt_scale_parameter
        )

        tmc_scale = F.softplus(
            self.tmc_scale_parameter
        )

        calibrated_three_mt_evidence = (
            three_mt_scale
            * three_mt_evidence
        )

        calibrated_tmc_evidence = (
            tmc_scale
            * tmc_evidence
        )

        three_mt_uncertainty = joint_opinion[
            "uncertainty"
        ]

        tmc_uncertainty = tmc_output[
            "uncertainty"
        ]

        mean_tmc_conflict = (
            self._calculate_mean_available_conflict(
                tmc_output=tmc_output,
                branch_masks=branch_masks,
                modality_order=modality_order,
            )
        )

        available_modality_count = (
            branch_masks.sum(
                dim=-1,
                keepdim=True,
            )
        )

        available_modality_proportion = (
            available_modality_count
            / float(
                self.number_of_modalities
            )
        )

        three_mt_weight = torch.full_like(
            three_mt_uncertainty,
            fill_value=0.5,
        )

        tmc_weight = torch.full_like(
            tmc_uncertainty,
            fill_value=0.5,
        )

        # These compatibility fields preserve the prediction-table
        # contract used by the learned-gate experiment.
        gate_logit = torch.zeros_like(
            three_mt_weight
        )

        gate_input = torch.cat(
            [
                three_mt_uncertainty.detach(),
                tmc_uncertainty.detach(),
                mean_tmc_conflict.detach(),
                available_modality_proportion,
                branch_masks,
            ],
            dim=-1,
        )

        final_evidence = (
            three_mt_weight
            * calibrated_three_mt_evidence
            +
            tmc_weight
            * calibrated_tmc_evidence
        )

        final_alpha = final_evidence + 1.0

        final_strength = final_alpha.sum(
            dim=-1,
            keepdim=True,
        )

        final_probabilities = (
            final_alpha
            / final_strength
        )

        final_uncertainty = (
            self.number_of_classes
            / final_strength
        )

        return {
            "evidence": final_evidence,
            "alpha": final_alpha,
            "strength": final_strength,
            "probabilities": final_probabilities,
            "uncertainty": final_uncertainty,
            "three_mt_weight": three_mt_weight,
            "tmc_weight": tmc_weight,
            "gate_logit": gate_logit,
            "gate_input": gate_input,
            "mean_tmc_conflict": mean_tmc_conflict,
            "available_modality_count": available_modality_count,
            "three_mt_scale": three_mt_scale,
            "tmc_scale": tmc_scale,
            "calibrated_three_mt_evidence":
                calibrated_three_mt_evidence,
            "calibrated_tmc_evidence":
                calibrated_tmc_evidence,
        }


## 7. Complete fixed-equal 3MT–TMC model


In [ ]:
# 7. Complete searchable fixed-equal 3MT–TMC model

class ADNIEvidential3MTTMCModel(nn.Module):
    def __init__(
        self,
        config,
    ):
        super().__init__()

        embedding_dim = int(
            config[
                "embedding_dim"
            ]
        )

        cmt_heads = int(
            config[
                "cmt_heads"
            ]
        )

        tabular_width_scale = float(
            config[
                "tabular_width_scale"
            ]
        )

        dropout_scale = float(
            config[
                "dropout_scale"
            ]
        )

        evidential_hidden_dim = int(
            config[
                "evidential_hidden_dim"
            ]
        )

        modality_dropout_probability = float(
            config[
                "modality_dropout_probability"
            ]
        )

        self.embedding_dim = (
            embedding_dim
        )

        self.number_of_classes = (
            NUMBER_OF_CLASSES
        )

        self.modality_order = list(
            MODALITY_ORDER
        )

        self.cascade_order = list(
            THREE_MT_CASCADE_ORDER
        )

        self.number_of_modalities = len(
            self.modality_order
        )

        self.modality_dropout_probability = (
            modality_dropout_probability
        )

        cmt_dropout = min(
            0.50,
            0.10
            * dropout_scale,
        )

        head_dropout = min(
            0.50,
            0.10
            * dropout_scale,
        )

        self.modality_encoders = (
            MaskedADNIModalityEncoders(
                output_dim=
                    embedding_dim,
                tabular_width_scale=
                    tabular_width_scale,
                dropout_scale=
                    dropout_scale,
            )
        )

        self.independent_evidential_heads = (
            IndependentModalityEvidentialHeads(
                modality_order=
                    self.modality_order,
                input_dim=
                    embedding_dim,
                number_of_classes=
                    NUMBER_OF_CLASSES,
                hidden_dim=
                    evidential_hidden_dim,
                dropout=
                    head_dropout,
            )
        )

        self.tmc_fusion = TMCFusion(
            number_of_classes=
                NUMBER_OF_CLASSES,
            modality_order=
                self.modality_order,
        )

        self.three_mt_cascade = (
            ThreeMTCascade(
                embedding_dim=
                    embedding_dim,
                modality_order=
                    self.modality_order,
                cascade_order=
                    self.cascade_order,
                number_of_heads=
                    cmt_heads,
                dropout=
                    cmt_dropout,
            )
        )

        self.three_mt_prediction_heads = (
            ThreeMTPredictionHeads(
                cascade_order=
                    self.cascade_order,
                embedding_dim=
                    embedding_dim,
                number_of_classes=
                    NUMBER_OF_CLASSES,
                evidential_hidden_dim=
                    evidential_hidden_dim,
                dropout=
                    head_dropout,
            )
        )

        self.hybrid_fusion = (
            FixedEqualHybridFusion(
                number_of_classes=
                    NUMBER_OF_CLASSES,
                number_of_modalities=
                    self.number_of_modalities,
                initial_three_mt_scale=
                    1.0,
                initial_tmc_scale=
                    1.0,
            )
        )

    def _apply_modality_dropout(
        self,
        original_branch_masks,
    ):
        if (
            not self.training
            or
            self.modality_dropout_probability
            <= 0.0
        ):
            return (
                original_branch_masks.clone(),
                torch.zeros_like(
                    original_branch_masks
                ),
            )

        retention_probability = (
            1.0
            - self.modality_dropout_probability
        )

        retention_masks = torch.bernoulli(
            torch.full_like(
                original_branch_masks,
                fill_value=
                    retention_probability,
            )
        )

        effective_branch_masks = (
            original_branch_masks
            * retention_masks
        )

        batch_size = (
            original_branch_masks.shape[
                0
            ]
        )

        for batch_row in range(
            batch_size
        ):
            available_indices = torch.nonzero(
                original_branch_masks[
                    batch_row
                ]
                > 0,
                as_tuple=False,
            ).flatten()

            if (
                effective_branch_masks[
                    batch_row
                ].sum()
                == 0
                and
                available_indices.numel()
                > 0
            ):
                selected_position = torch.randint(
                    low=0,
                    high=
                        available_indices.numel(),
                    size=(1,),
                    device=
                        original_branch_masks.device,
                )

                selected_modality_index = (
                    available_indices[
                        selected_position
                    ].item()
                )

                effective_branch_masks[
                    batch_row,
                    selected_modality_index,
                ] = 1.0

        dropped_branch_masks = (
            original_branch_masks
            - effective_branch_masks
        ).clamp(
            min=0.0,
            max=1.0,
        )

        return (
            effective_branch_masks,
            dropped_branch_masks,
        )

    def _replace_branch_masks(
        self,
        modalities,
        effective_branch_masks,
    ):
        effective_modalities = {}

        for modality_index, modality_name in enumerate(
            self.modality_order
        ):
            effective_modalities[
                modality_name
            ] = dict(
                modalities[
                    modality_name
                ]
            )

            effective_modalities[
                modality_name
            ][
                "branch_mask"
            ] = (
                effective_branch_masks[
                    :,
                    modality_index,
                ]
            )

        return effective_modalities

    def forward(
        self,
        modalities,
        original_branch_masks,
    ):
        (
            effective_branch_masks,
            dropped_branch_masks,
        ) = self._apply_modality_dropout(
            original_branch_masks
        )

        effective_modalities = (
            self._replace_branch_masks(
                modalities=
                    modalities,
                effective_branch_masks=
                    effective_branch_masks,
            )
        )

        encoded_modalities = (
            self.modality_encoders(
                effective_modalities
            )
        )

        masked_representations = (
            encoded_modalities[
                "masked"
            ]
        )

        modality_opinions = (
            self.independent_evidential_heads(
                modality_representations=
                    masked_representations,
                modalities=
                    effective_modalities,
            )
        )

        tmc_output = self.tmc_fusion(
            modality_opinions=
                modality_opinions
        )

        three_mt_output = (
            self.three_mt_cascade(
                masked_representations=
                    masked_representations,
                branch_masks=
                    effective_branch_masks,
            )
        )

        three_mt_predictions = (
            self.three_mt_prediction_heads(
                three_mt_output=
                    three_mt_output
            )
        )

        joint_opinion = (
            three_mt_predictions[
                "joint_opinion"
            ]
        )

        final_output = (
            self.hybrid_fusion(
                joint_opinion=
                    joint_opinion,
                tmc_output=
                    tmc_output,
                branch_masks=
                    effective_branch_masks,
                modality_order=
                    self.modality_order,
            )
        )

        return {
            "final_output":
                final_output,
            "modality_opinions":
                modality_opinions,
            "tmc_output":
                tmc_output,
            "three_mt_output":
                three_mt_output,
            "three_mt_predictions":
                three_mt_predictions,
            "joint_opinion":
                joint_opinion,
            "encoded_modalities":
                encoded_modalities,
            "original_branch_masks":
                original_branch_masks,
            "effective_branch_masks":
                effective_branch_masks,
            "dropped_branch_masks":
                dropped_branch_masks,
        }

def count_trainable_parameters(
    module,
):
    return sum(
        parameter.numel()
        for parameter
        in module.parameters()
        if parameter.requires_grad
    )


## 8. Validate all five locked selections **before touching test data**

The standard fold 1–4 location is checked first. For Fold 0, or if a standard path is absent, the notebook searches under the model's `experiments` directory for a locked manifest whose JSON says `outer_fold == fold`.

If more than one non-standard match exists, the notebook stops instead of guessing. Set `LOCKED_MANIFEST_OVERRIDES[fold]` to the intended file in that case.


In [ ]:
# 8. Locked-manifest discovery and validation

REQUIRED_MODEL_CONFIG_KEYS = {
    "embedding_dim",
    "cmt_heads",
    "tabular_width_scale",
    "dropout_scale",
    "evidential_hidden_dim",
    "modality_dropout_probability",
}

def _load_json(path):
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)

def _canonical_path(path_like):
    return Path(str(path_like)).expanduser()

def locate_locked_manifest(outer_fold):
    override = LOCKED_MANIFEST_OVERRIDES.get(outer_fold)

    if override:
        path = _canonical_path(override)
        if not path.is_file():
            raise FileNotFoundError(
                f"Fold {outer_fold} override does not exist: {path}"
            )
        payload = _load_json(path)
        if int(payload.get("outer_fold", -1)) != int(outer_fold):
            raise ValueError(
                f"Override {path} does not describe outer fold {outer_fold}."
            )
        return path, payload

    standard = (
        SELECTION_ROOT
        / f"fold_{outer_fold}"
        / "locked_selection"
        / "locked_selection.json"
    )

    if standard.is_file():
        payload = _load_json(standard)
        if int(payload.get("outer_fold", -1)) == int(outer_fold):
            return standard, payload

    matches = []
    for candidate in EXPERIMENTS_ROOT.rglob("locked_selection.json"):
        try:
            payload = _load_json(candidate)
        except Exception:
            continue

        if int(payload.get("outer_fold", -1)) == int(outer_fold):
            matches.append((candidate, payload))

    # De-duplicate paths in case a mounted filesystem exposes aliases.
    unique = {}
    for path, payload in matches:
        unique[str(path)] = (path, payload)
    matches = list(unique.values())

    if len(matches) == 1:
        return matches[0]

    if len(matches) == 0:
        raise FileNotFoundError(
            f"No locked_selection.json found for outer fold {outer_fold}. "
            "Set LOCKED_MANIFEST_OVERRIDES for this fold if it is stored "
            "outside MODEL_ROOT/experiments."
        )

    candidate_text = "\n".join(f"  - {p}" for p, _ in matches)
    raise RuntimeError(
        f"Multiple locked manifests found for outer fold {outer_fold}. "
        "Set LOCKED_MANIFEST_OVERRIDES to the intended one:\n"
        f"{candidate_text}"
    )

def selected_checkpoint_entries(payload):
    entries = payload.get("selected_checkpoints")
    if not isinstance(entries, list) or not entries:
        raise ValueError(
            "Locked manifest has no non-empty 'selected_checkpoints' list."
        )
    return entries

def resolve_checkpoint_path(manifest_path, payload, entry):
    seed = int(entry["seed"])
    raw = Path(str(entry.get("checkpoint_path", "")))

    candidates = []
    if str(raw):
        candidates.append(raw)

    # If the Google Drive prefix changed but the path from MODEL_ROOT onward
    # stayed the same, reconstruct it under the current mounted MODEL_ROOT.
    marker = "models/3mt_tmc_evidential/"
    raw_text = str(raw).replace("\\", "/")
    if marker in raw_text:
        relative_tail = raw_text.split(marker, 1)[1]
        candidates.append(MODEL_ROOT / relative_tail)

    selected_name = payload.get("selected_configuration_name")
    fold_root = manifest_path.parent.parent

    if selected_name:
        candidates.append(
            fold_root
            / "five_seed_confirmation"
            / str(selected_name)
            / f"seed_{seed}"
            / "best_validation_auc_checkpoint.pt"
        )

    for candidate in candidates:
        if candidate.is_file():
            return candidate

    # Last-resort local search around this fold's experiment directory.
    local_matches = []
    if fold_root.is_dir():
        for candidate in fold_root.rglob("best_validation_auc_checkpoint.pt"):
            if f"seed_{seed}" in candidate.parts:
                local_matches.append(candidate)

    if len(local_matches) == 1:
        return local_matches[0]

    raise FileNotFoundError(
        f"Could not resolve checkpoint for fold={payload.get('outer_fold')}, "
        f"seed={seed}. Manifest entry was: {entry}"
    )

def normalize_config_value(value):
    if isinstance(value, np.generic):
        return value.item()
    return value

def load_checkpoint_safely(path, map_location="cpu"):
    # PyTorch versions differ in the default behavior of weights_only.
    try:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=map_location,
        )

locked_folds = {}
manifest_summary_rows = []

for fold in OUTER_FOLDS:
    manifest_path, payload = locate_locked_manifest(fold)

    if int(payload.get("outer_fold", -1)) != fold:
        raise ValueError(
            f"Manifest outer_fold mismatch for fold {fold}: {manifest_path}"
        )

    if payload.get("selection_locked_before_test") is not True:
        raise ValueError(
            f"Fold {fold} does not explicitly confirm selection_locked_before_test=True."
        )

    if payload.get("test_role_scored") is True:
        raise ValueError(
            f"Fold {fold} manifest says test_role_scored=True; stopping."
        )

    config = payload.get("selected_config")
    if not isinstance(config, dict):
        raise ValueError(
            f"Fold {fold} manifest has no selected_config dictionary."
        )

    missing_config = REQUIRED_MODEL_CONFIG_KEYS.difference(config)
    if missing_config:
        raise ValueError(
            f"Fold {fold} selected_config is missing: {sorted(missing_config)}"
        )

    entries = selected_checkpoint_entries(payload)
    entry_seeds = sorted(int(item["seed"]) for item in entries)

    if entry_seeds != sorted(EXPECTED_SEEDS):
        raise ValueError(
            f"Fold {fold} selected checkpoint seeds are {entry_seeds}, "
            f"expected {sorted(EXPECTED_SEEDS)}."
        )

    resolved_checkpoints = {}
    checkpoint_metadata = {}

    for entry in entries:
        seed = int(entry["seed"])
        checkpoint_path = resolve_checkpoint_path(
            manifest_path=manifest_path,
            payload=payload,
            entry=entry,
        )

        checkpoint = load_checkpoint_safely(
            checkpoint_path,
            map_location="cpu",
        )

        if not isinstance(checkpoint, dict) or "model_state_dict" not in checkpoint:
            raise ValueError(
                f"Unexpected checkpoint format: {checkpoint_path}"
            )

        if "outer_fold" in checkpoint and int(checkpoint["outer_fold"]) != fold:
            raise ValueError(
                f"Checkpoint fold mismatch: {checkpoint_path}"
            )

        if "seed" in checkpoint and int(checkpoint["seed"]) != seed:
            raise ValueError(
                f"Checkpoint seed mismatch: {checkpoint_path}"
            )

        checkpoint_config = checkpoint.get("config")
        if isinstance(checkpoint_config, dict):
            for key in REQUIRED_MODEL_CONFIG_KEYS:
                left = normalize_config_value(config[key])
                right = normalize_config_value(checkpoint_config[key])
                if left != right:
                    raise ValueError(
                        f"Fold {fold}, seed {seed}: selected_config and "
                        f"checkpoint config disagree for '{key}': {left} vs {right}"
                    )

        resolved_checkpoints[seed] = checkpoint_path
        checkpoint_metadata[seed] = {
            "best_epoch": checkpoint.get("best_epoch"),
            "best_validation_roc_auc": checkpoint.get("best_validation_roc_auc"),
        }

        del checkpoint

    locked_folds[fold] = {
        "manifest_path": manifest_path,
        "payload": payload,
        "config": config,
        "checkpoints": resolved_checkpoints,
        "checkpoint_metadata": checkpoint_metadata,
    }

    validation_summary = payload.get("selected_validation_summary", {})
    manifest_summary_rows.append({
        "outer_fold": fold,
        "selected_configuration": payload.get("selected_configuration_name"),
        "mean_validation_roc_auc": validation_summary.get("mean_validation_roc_auc"),
        "sd_validation_roc_auc": validation_summary.get("sd_validation_roc_auc"),
        "manifest_path": str(manifest_path),
    })

manifest_summary = pd.DataFrame(manifest_summary_rows).sort_values("outer_fold")

display(manifest_summary)

print("\n" + "=" * 78)
print("ALL FIVE FOLD SELECTIONS VERIFIED")
print("=" * 78)
print("No test table or test DataLoader has been loaded yet.")


## 9. Freeze a reproducibility snapshot of the locked selections

This snapshot is written **before** any test inference. It records exactly which manifests, configurations, and checkpoints are about to be evaluated.


In [ ]:
# 9. Save pre-test lock snapshot

lock_snapshot = {}
for fold, info in locked_folds.items():
    payload = info["payload"]
    lock_snapshot[str(fold)] = {
        "manifest_path": str(info["manifest_path"]),
        "selected_configuration_name": payload.get("selected_configuration_name"),
        "selected_config": payload.get("selected_config"),
        "selected_validation_summary": payload.get("selected_validation_summary"),
        "checkpoints": {
            str(seed): str(path)
            for seed, path in sorted(info["checkpoints"].items())
        },
    }

lock_signature = hashlib.sha256(
    json.dumps(
        lock_snapshot,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()

snapshot_payload = {
    "evaluation": "final nested-CV outer-test evaluation",
    "outer_folds": OUTER_FOLDS,
    "seeds": EXPECTED_SEEDS,
    "aggregation": {
        "checkpoint_level": "evaluate all five locked seeds separately",
        "seed_ensemble": "arithmetic mean of class probabilities across the five locked seeds",
        "oof": "concatenate one seed-ensemble test prediction per subject across outer folds",
    },
    "lock_signature": lock_signature,
    "locked_folds": lock_snapshot,
}

with open(
    OUTPUT_ROOT / "pre_test_locked_selection_snapshot.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(snapshot_payload, handle, indent=2)

print(f"Lock signature: {lock_signature}")
print(
    "Saved: "
    f"{OUTPUT_ROOT / 'pre_test_locked_selection_snapshot.json'}"
)


## 10. Build test DataLoaders

All model selections are now verified and snapshotted. This is the first cell that accesses the `test` role.


In [ ]:
# 10. Test-table and DataLoader construction

def set_global_random_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

def fold_input_path(outer_fold):
    return (
        INPUT_DIR
        / f"mci_prognosis_outer_fold_{outer_fold}_final_task_ready.csv"
    )

def load_fold_table(outer_fold):
    path = fold_input_path(outer_fold)
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing prepared fold input: {path}"
        )

    table = pd.read_csv(path)
    expected_roles = {"train", "validation", "test"}
    observed_roles = set(table["DATA_ROLE"].dropna().unique())

    if not expected_roles.issubset(observed_roles):
        raise ValueError(
            f"Unexpected role structure in fold {outer_fold}: {observed_roles}"
        )
    return table

def build_test_loader(outer_fold):
    fold_table = load_fold_table(outer_fold)
    test_table = (
        fold_table.loc[
            fold_table["DATA_ROLE"] == "test"
        ]
        .reset_index(drop=True)
        .copy()
    )

    if test_table.empty:
        raise ValueError(f"Fold {outer_fold} has an empty test role.")

    if test_table[target_column].isna().any():
        raise ValueError(f"Fold {outer_fold} test role has missing targets.")

    if test_table["RID"].duplicated().any():
        duplicated = test_table.loc[
            test_table["RID"].duplicated(keep=False),
            ["RID", "PTID"],
        ]
        raise ValueError(
            f"Fold {outer_fold} test role contains duplicate RID values:\n{duplicated}"
        )

    test_dataset = ADNIMultimodalDataset(
        dataframe=test_table,
        column_contract=dataset_column_contract,
        branch_mask_order=branch_mask_columns,
        feature_mask_order=feature_mask_columns,
        target_column=target_column,
        load_mri=True,
    )

    test_loader = DataLoader(
        dataset=test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )

    return {
        "test_table": test_table,
        "test_loader": test_loader,
    }

test_resources = {}
for fold in OUTER_FOLDS:
    resources = build_test_loader(fold)
    test_resources[fold] = resources
    counts = resources["test_table"][target_column].value_counts().sort_index().to_dict()
    print(
        f"Fold {fold}: n_test={len(resources['test_table'])}, "
        f"class_counts={counts}"
    )


## 11. Inference helpers

For every locked checkpoint, this saves:

- RID / PTID and true label;
- 3MT pMCI probability;
- TMC pMCI probability;
- final fixed-equal hybrid pMCI probability;
- 3MT, TMC, and final evidential uncertainty;
- mean TMC conflict;
- learned pathway evidence scales;
- availability masks and available-modality count.


In [ ]:
# 11. Inference and metric helpers

def move_nested_to_device(value, device):
    if isinstance(value, torch.Tensor):
        return value.to(device, non_blocking=True)
    if isinstance(value, dict):
        return {
            key: move_nested_to_device(item, device)
            for key, item in value.items()
        }
    if isinstance(value, list):
        return [move_nested_to_device(item, device) for item in value]
    if isinstance(value, tuple):
        return tuple(move_nested_to_device(item, device) for item in value)
    return value

def scalar_from_tensor(value):
    if isinstance(value, torch.Tensor):
        value = value.detach().cpu().reshape(-1)
        if value.numel() != 1:
            raise ValueError("Expected scalar tensor.")
        return float(value.item())
    return float(value)

def prediction_file_path(outer_fold, seed):
    run_dir = SEED_PREDICTION_ROOT / f"fold_{outer_fold}"
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir / f"seed_{seed}.csv"

def completed_prediction_is_valid(path, test_table, outer_fold, seed):
    if not path.is_file():
        return False

    try:
        frame = pd.read_csv(path)
    except Exception:
        return False

    required = {
        "outer_fold", "seed", "RID", "PTID", "target",
        "p_3mt", "p_tmc", "p_hybrid",
        "uncertainty_3mt", "uncertainty_tmc", "uncertainty_hybrid",
        "mean_tmc_conflict",
    }
    if not required.issubset(frame.columns):
        return False

    if len(frame) != len(test_table):
        return False

    if set(frame["outer_fold"].astype(int).unique()) != {int(outer_fold)}:
        return False

    if set(frame["seed"].astype(int).unique()) != {int(seed)}:
        return False

    expected_ids = set(test_table["RID"].astype(int).tolist())
    observed_ids = set(frame["RID"].astype(int).tolist())
    return expected_ids == observed_ids

@torch.inference_mode()
def predict_one_checkpoint(
    outer_fold,
    seed,
    config,
    checkpoint_path,
    loader,
):
    set_global_random_seed(seed)

    checkpoint = load_checkpoint_safely(
        checkpoint_path,
        map_location="cpu",
    )

    model = ADNIEvidential3MTTMCModel(
        config=config
    ).to(DEVICE)

    model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True,
    )
    model.eval()

    rows = []

    for batch in loader:
        # Preserve CPU metadata before moving nested tensors to DEVICE.
        rid_values = batch["rid"].detach().cpu().numpy().astype(int)
        target_values = batch["target"].detach().cpu().numpy().astype(int)
        ptid_values = list(batch["ptid"])
        branch_masks_cpu = batch["branch_masks"].detach().cpu().numpy()

        batch_device = move_nested_to_device(batch, DEVICE)

        output = model(
            modalities=batch_device["modalities"],
            original_branch_masks=batch_device["branch_masks"],
        )

        joint = output["joint_opinion"]
        tmc = output["tmc_output"]
        final = output["final_output"]

        p_3mt = joint["probabilities"][:, 1].detach().cpu().numpy()
        p_tmc = tmc["probabilities"][:, 1].detach().cpu().numpy()
        p_hybrid = final["probabilities"][:, 1].detach().cpu().numpy()

        u_3mt = joint["uncertainty"].squeeze(-1).detach().cpu().numpy()
        u_tmc = tmc["uncertainty"].squeeze(-1).detach().cpu().numpy()
        u_hybrid = final["uncertainty"].squeeze(-1).detach().cpu().numpy()
        mean_conflict = final["mean_tmc_conflict"].squeeze(-1).detach().cpu().numpy()
        available_count = final["available_modality_count"].squeeze(-1).detach().cpu().numpy()

        three_mt_scale = scalar_from_tensor(final["three_mt_scale"])
        tmc_scale = scalar_from_tensor(final["tmc_scale"])

        for i in range(len(rid_values)):
            row = {
                "outer_fold": int(outer_fold),
                "seed": int(seed),
                "RID": int(rid_values[i]),
                "PTID": str(ptid_values[i]),
                "target": int(target_values[i]),
                "p_3mt": float(p_3mt[i]),
                "p_tmc": float(p_tmc[i]),
                "p_hybrid": float(p_hybrid[i]),
                "uncertainty_3mt": float(u_3mt[i]),
                "uncertainty_tmc": float(u_tmc[i]),
                "uncertainty_hybrid": float(u_hybrid[i]),
                "mean_tmc_conflict": float(mean_conflict[i]),
                "available_modality_count": float(available_count[i]),
                "three_mt_scale": float(three_mt_scale),
                "tmc_scale": float(tmc_scale),
            }

            for modality_index, modality_name in enumerate(MODALITY_ORDER):
                row[f"available_{modality_name}"] = float(
                    branch_masks_cpu[i, modality_index]
                )

            rows.append(row)

    frame = pd.DataFrame(rows)

    del model
    del checkpoint
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return frame

def binary_probability_metrics(targets, probabilities):
    targets = np.asarray(targets, dtype=np.int64)
    probabilities = np.asarray(probabilities, dtype=np.float64)

    if np.unique(targets).size < 2:
        return {
            "roc_auc": np.nan,
            "average_precision": np.nan,
            "brier_score": np.nan,
        }

    return {
        "roc_auc": float(roc_auc_score(targets, probabilities)),
        "average_precision": float(average_precision_score(targets, probabilities)),
        "brier_score": float(brier_score_loss(targets, probabilities)),
    }

PATHWAY_COLUMNS = {
    "3MT": "p_3mt",
    "TMC": "p_tmc",
    "Hybrid fixed-equal": "p_hybrid",
}


## 12. Run/resume all 25 locked checkpoint evaluations

This is the cell that performs outer-test inference. Each fold uses only the five checkpoints frozen in its own locked manifest.


In [ ]:
# 12. Final outer-test inference

all_seed_prediction_frames = []

for fold in OUTER_FOLDS:
    info = locked_folds[fold]
    resources = test_resources[fold]
    test_table = resources["test_table"]
    test_loader = resources["test_loader"]

    print("\n" + "=" * 78)
    print(f"OUTER FOLD {fold}")
    print("=" * 78)
    print(
        f"Selected configuration: "
        f"{info['payload'].get('selected_configuration_name')}"
    )

    for seed in EXPECTED_SEEDS:
        save_path = prediction_file_path(fold, seed)

        if (
            RESUME_COMPLETED_CHECKPOINTS
            and completed_prediction_is_valid(
                save_path,
                test_table=test_table,
                outer_fold=fold,
                seed=seed,
            )
        ):
            print(f"Loading completed test inference: fold={fold}, seed={seed}")
            predictions = pd.read_csv(save_path)
        else:
            print(f"Evaluating locked checkpoint: fold={fold}, seed={seed}")
            predictions = predict_one_checkpoint(
                outer_fold=fold,
                seed=seed,
                config=info["config"],
                checkpoint_path=info["checkpoints"][seed],
                loader=test_loader,
            )
            predictions.to_csv(save_path, index=False)

        all_seed_prediction_frames.append(predictions)

seed_predictions = pd.concat(
    all_seed_prediction_frames,
    ignore_index=True,
)

seed_predictions.to_csv(
    OUTPUT_ROOT / "all_seed_level_test_predictions.csv",
    index=False,
)

print("\nSaved all seed-level predictions:")
print(OUTPUT_ROOT / "all_seed_level_test_predictions.csv")
print(f"Rows: {len(seed_predictions):,}")


## 13. Checkpoint-level test metrics

These metrics treat the five seeds as locked replicate models. Nothing is selected based on these results.


In [ ]:
# 13. Per-fold, per-seed, per-pathway test metrics

seed_metric_rows = []

for (fold, seed), group in seed_predictions.groupby(
    ["outer_fold", "seed"],
    sort=True,
):
    for pathway, probability_column in PATHWAY_COLUMNS.items():
        metrics = binary_probability_metrics(
            group["target"],
            group[probability_column],
        )
        seed_metric_rows.append({
            "outer_fold": int(fold),
            "seed": int(seed),
            "pathway": pathway,
            "n_test": int(len(group)),
            **metrics,
        })

seed_level_metrics = pd.DataFrame(seed_metric_rows)
seed_level_metrics.to_csv(
    OUTPUT_ROOT / "seed_level_test_metrics.csv",
    index=False,
)

display(seed_level_metrics)

# Summary of replicate performance across all 25 fold×seed test evaluations.
seed_metric_summary = (
    seed_level_metrics
    .groupby("pathway")
    .agg(
        mean_roc_auc=("roc_auc", "mean"),
        sd_roc_auc=("roc_auc", "std"),
        mean_average_precision=("average_precision", "mean"),
        sd_average_precision=("average_precision", "std"),
        mean_brier_score=("brier_score", "mean"),
        sd_brier_score=("brier_score", "std"),
    )
    .reset_index()
)

seed_metric_summary.to_csv(
    OUTPUT_ROOT / "seed_level_metric_summary.csv",
    index=False,
)

display(seed_metric_summary)


## 14. Pre-specified five-seed ensemble within each outer fold

For each test participant, probabilities are averaged across the five locked seeds. Uncertainty and conflict quantities are also averaged for descriptive/error-detection analysis. No seed is chosen or discarded.


In [ ]:
# 14. Build one seed-ensemble prediction per test subject

expected_seed_count = len(EXPECTED_SEEDS)
seed_counts = (
    seed_predictions
    .groupby(["outer_fold", "RID"])["seed"]
    .nunique()
)

if not (seed_counts == expected_seed_count).all():
    bad = seed_counts[seed_counts != expected_seed_count]
    raise ValueError(
        "Some test subjects do not have exactly five seed predictions:\n"
        f"{bad.head(20)}"
    )

# Verify subject metadata/availability is invariant across the five seeds.
invariant_columns = [
    "PTID",
    "target",
    "available_modality_count",
] + [f"available_{name}" for name in MODALITY_ORDER]

for column in invariant_columns:
    nunique = (
        seed_predictions
        .groupby(["outer_fold", "RID"])[column]
        .nunique(dropna=False)
    )
    if (nunique > 1).any():
        raise ValueError(
            f"Column '{column}' changes across seeds for the same test subject."
        )

aggregation = {
    "PTID": "first",
    "target": "first",
    "p_3mt": "mean",
    "p_tmc": "mean",
    "p_hybrid": "mean",
    "uncertainty_3mt": "mean",
    "uncertainty_tmc": "mean",
    "uncertainty_hybrid": "mean",
    "mean_tmc_conflict": "mean",
    "available_modality_count": "first",
    "three_mt_scale": "mean",
    "tmc_scale": "mean",
}
for modality_name in MODALITY_ORDER:
    aggregation[f"available_{modality_name}"] = "first"

ensemble_predictions = (
    seed_predictions
    .groupby(["outer_fold", "RID"], as_index=False)
    .agg(aggregation)
)

# Seed-to-seed predictive variability is retained as an additional uncertainty
# descriptor, but it is not used to alter predictions.
seed_variability = (
    seed_predictions
    .groupby(["outer_fold", "RID"])
    .agg(
        sd_p_3mt=("p_3mt", "std"),
        sd_p_tmc=("p_tmc", "std"),
        sd_p_hybrid=("p_hybrid", "std"),
    )
    .reset_index()
)

ensemble_predictions = ensemble_predictions.merge(
    seed_variability,
    on=["outer_fold", "RID"],
    how="left",
    validate="one_to_one",
)

ensemble_predictions["predicted_label_hybrid"] = (
    ensemble_predictions["p_hybrid"] >= 0.5
).astype(int)
ensemble_predictions["hybrid_error"] = (
    ensemble_predictions["predicted_label_hybrid"]
    != ensemble_predictions["target"]
).astype(int)

ensemble_predictions.to_csv(
    OUTPUT_ROOT / "fold_seed_ensemble_predictions.csv",
    index=False,
)

display(ensemble_predictions.head())
print(f"Seed-ensemble rows: {len(ensemble_predictions):,}")


## 15. Fold-level ensemble metrics and pooled OOF metrics

Two complementary summaries are saved:

- **fold-level nested-CV metrics:** evaluate each outer fold separately, then summarize across the five folds;
- **pooled OOF metrics:** concatenate the five seed-ensemble test folds and evaluate one OOF probability per participant.


In [ ]:
# 15. Fold-level and pooled OOF evaluation

fold_ensemble_metric_rows = []

for fold, group in ensemble_predictions.groupby("outer_fold", sort=True):
    for pathway, probability_column in PATHWAY_COLUMNS.items():
        metrics = binary_probability_metrics(
            group["target"],
            group[probability_column],
        )
        fold_ensemble_metric_rows.append({
            "outer_fold": int(fold),
            "pathway": pathway,
            "n_test": int(len(group)),
            **metrics,
        })

fold_ensemble_metrics = pd.DataFrame(fold_ensemble_metric_rows)
fold_ensemble_metrics.to_csv(
    OUTPUT_ROOT / "fold_seed_ensemble_metrics.csv",
    index=False,
)

display(fold_ensemble_metrics)

fold_metric_summary = (
    fold_ensemble_metrics
    .groupby("pathway")
    .agg(
        mean_fold_roc_auc=("roc_auc", "mean"),
        sd_fold_roc_auc=("roc_auc", "std"),
        mean_fold_average_precision=("average_precision", "mean"),
        sd_fold_average_precision=("average_precision", "std"),
        mean_fold_brier_score=("brier_score", "mean"),
        sd_fold_brier_score=("brier_score", "std"),
    )
    .reset_index()
)

fold_metric_summary.to_csv(
    OUTPUT_ROOT / "fold_metric_summary.csv",
    index=False,
)

# OOF integrity: every participant must occur in exactly one outer-test fold.
if ensemble_predictions["RID"].duplicated().any():
    duplicates = ensemble_predictions.loc[
        ensemble_predictions["RID"].duplicated(keep=False),
        ["RID", "PTID", "outer_fold"],
    ].sort_values(["RID", "outer_fold"])
    raise ValueError(
        "A participant appears in more than one outer-test fold:\n"
        f"{duplicates.head(30)}"
    )

pooled_oof_predictions = ensemble_predictions.sort_values(
    ["outer_fold", "RID"]
).reset_index(drop=True)

pooled_oof_predictions.to_csv(
    OUTPUT_ROOT / "pooled_oof_predictions.csv",
    index=False,
)

pooled_metric_rows = []
for pathway, probability_column in PATHWAY_COLUMNS.items():
    metrics = binary_probability_metrics(
        pooled_oof_predictions["target"],
        pooled_oof_predictions[probability_column],
    )
    pooled_metric_rows.append({
        "pathway": pathway,
        "n_oof": int(len(pooled_oof_predictions)),
        **metrics,
    })

pooled_oof_metrics = pd.DataFrame(pooled_metric_rows)
pooled_oof_metrics.to_csv(
    OUTPUT_ROOT / "pooled_oof_metrics.csv",
    index=False,
)

print("\n" + "=" * 78)
print("FOLD-LEVEL SEED-ENSEMBLE SUMMARY")
print("=" * 78)
display(fold_metric_summary)

print("\n" + "=" * 78)
print("POOLED OUT-OF-FOLD METRICS")
print("=" * 78)
display(pooled_oof_metrics)

print(
    f"OOF participants: {len(pooled_oof_predictions):,} | "
    f"unique RID: {pooled_oof_predictions['RID'].nunique():,}"
)


## 16. Pooled seed-replicate summary

This preserves the five locked seeds as replicates across the full nested-CV procedure. For each seed, the notebook concatenates that seed's test predictions across all five folds, then reports the mean ± SD of those five pooled-seed metrics.


In [ ]:
# 16. Pooled OOF metrics separately for each locked seed

pooled_seed_metric_rows = []

for seed, group in seed_predictions.groupby("seed", sort=True):
    if group["RID"].duplicated().any():
        raise ValueError(
            f"Seed {seed}: duplicate RID across outer-test folds."
        )

    for pathway, probability_column in PATHWAY_COLUMNS.items():
        metrics = binary_probability_metrics(
            group["target"],
            group[probability_column],
        )
        pooled_seed_metric_rows.append({
            "seed": int(seed),
            "pathway": pathway,
            "n_oof": int(len(group)),
            **metrics,
        })

pooled_seed_metrics = pd.DataFrame(pooled_seed_metric_rows)
pooled_seed_metrics.to_csv(
    OUTPUT_ROOT / "pooled_oof_metrics_by_seed.csv",
    index=False,
)

pooled_seed_summary = (
    pooled_seed_metrics
    .groupby("pathway")
    .agg(
        mean_roc_auc=("roc_auc", "mean"),
        sd_roc_auc=("roc_auc", "std"),
        mean_average_precision=("average_precision", "mean"),
        sd_average_precision=("average_precision", "std"),
        mean_brier_score=("brier_score", "mean"),
        sd_brier_score=("brier_score", "std"),
    )
    .reset_index()
)

pooled_seed_summary.to_csv(
    OUTPUT_ROOT / "pooled_oof_seed_summary.csv",
    index=False,
)

display(pooled_seed_metrics)
display(pooled_seed_summary)


## 17. Uncertainty and conflict as error-detection signals

This evaluates each saved signal **as-is** against whether the seed-ensemble hybrid prediction is wrong. It does not tune a threshold or invent a weighted combination after seeing test outcomes.


In [ ]:
# 17. Error-detection ROC AUC for pre-existing uncertainty signals

error_targets = pooled_oof_predictions["hybrid_error"].to_numpy(dtype=np.int64)

ERROR_SIGNAL_COLUMNS = {
    "Hybrid evidential uncertainty": "uncertainty_hybrid",
    "3MT evidential uncertainty": "uncertainty_3mt",
    "TMC evidential uncertainty": "uncertainty_tmc",
    "Mean TMC conflict": "mean_tmc_conflict",
    "Seed SD of hybrid probability": "sd_p_hybrid",
}

error_detection_rows = []
for signal_name, column in ERROR_SIGNAL_COLUMNS.items():
    values = pooled_oof_predictions[column].to_numpy(dtype=np.float64)

    if np.unique(error_targets).size < 2 or np.all(values == values[0]):
        auc = np.nan
    else:
        auc = float(roc_auc_score(error_targets, values))

    error_detection_rows.append({
        "signal": signal_name,
        "error_detection_roc_auc": auc,
        "n_errors": int(error_targets.sum()),
        "n_correct": int((1 - error_targets).sum()),
    })

error_detection_metrics = pd.DataFrame(error_detection_rows)
error_detection_metrics.to_csv(
    OUTPUT_ROOT / "error_detection_metrics.csv",
    index=False,
)

display(error_detection_metrics)


## 18. Figures

These plots are descriptive only and are generated after all predictions are fixed.


In [ ]:
# 18. Pooled OOF ROC curves

plt.figure(figsize=(7, 6))

for pathway, probability_column in PATHWAY_COLUMNS.items():
    fpr, tpr, _ = roc_curve(
        pooled_oof_predictions["target"],
        pooled_oof_predictions[probability_column],
    )
    auc = roc_auc_score(
        pooled_oof_predictions["target"],
        pooled_oof_predictions[probability_column],
    )
    plt.plot(fpr, tpr, label=f"{pathway} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("Nested-CV pooled out-of-fold ROC")
plt.legend()
plt.tight_layout()
plt.savefig(
    OUTPUT_ROOT / "pooled_oof_roc_curves.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


In [ ]:
# 18.1 Hybrid fold-by-fold ROC AUC

hybrid_fold_metrics = fold_ensemble_metrics.loc[
    fold_ensemble_metrics["pathway"] == "Hybrid fixed-equal"
].sort_values("outer_fold")

plt.figure(figsize=(7, 4.5))
plt.bar(
    hybrid_fold_metrics["outer_fold"].astype(str),
    hybrid_fold_metrics["roc_auc"],
)
plt.ylim(0.5, 1.0)
plt.xlabel("Outer fold")
plt.ylabel("ROC AUC")
plt.title("Fixed-equal hybrid: seed-ensemble test ROC AUC by fold")
plt.tight_layout()
plt.savefig(
    OUTPUT_ROOT / "hybrid_fold_auc.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


## 19. Save final evaluation manifest

This file records the lock signature and all generated outputs. The test results should now be treated as final; model/hyperparameter selection should not be reopened based on them.


In [ ]:
# 19. Final evaluation manifest

hybrid_pooled_row = (
    pooled_oof_metrics.loc[
        pooled_oof_metrics["pathway"] == "Hybrid fixed-equal"
    ]
    .iloc[0]
    .to_dict()
)

hybrid_fold_summary_row = (
    fold_metric_summary.loc[
        fold_metric_summary["pathway"] == "Hybrid fixed-equal"
    ]
    .iloc[0]
    .to_dict()
)

final_evaluation_manifest = {
    "evaluation": "final nested-CV outer-test evaluation",
    "lock_signature": lock_signature,
    "outer_folds": OUTER_FOLDS,
    "seeds": EXPECTED_SEEDS,
    "n_unique_oof_participants": int(pooled_oof_predictions["RID"].nunique()),
    "primary_model": "availability-gated fixed-equal 3MT-TMC evidential model",
    "fusion_weights": {
        "three_mt": 0.5,
        "tmc": 0.5,
    },
    "aggregation": {
        "per_checkpoint": "all five locked seeds evaluated separately",
        "per_fold_seed_ensemble": "arithmetic mean of the five locked seed probabilities",
        "pooled_oof": "concatenation of one fold-specific seed-ensemble prediction per participant",
    },
    "hybrid_pooled_oof": {
        key: (value.item() if isinstance(value, np.generic) else value)
        for key, value in hybrid_pooled_row.items()
    },
    "hybrid_fold_summary": {
        key: (value.item() if isinstance(value, np.generic) else value)
        for key, value in hybrid_fold_summary_row.items()
    },
    "outputs": {
        "seed_level_predictions": str(OUTPUT_ROOT / "all_seed_level_test_predictions.csv"),
        "seed_level_metrics": str(OUTPUT_ROOT / "seed_level_test_metrics.csv"),
        "seed_ensemble_predictions": str(OUTPUT_ROOT / "fold_seed_ensemble_predictions.csv"),
        "fold_seed_ensemble_metrics": str(OUTPUT_ROOT / "fold_seed_ensemble_metrics.csv"),
        "pooled_oof_predictions": str(OUTPUT_ROOT / "pooled_oof_predictions.csv"),
        "pooled_oof_metrics": str(OUTPUT_ROOT / "pooled_oof_metrics.csv"),
        "pooled_oof_metrics_by_seed": str(OUTPUT_ROOT / "pooled_oof_metrics_by_seed.csv"),
        "error_detection_metrics": str(OUTPUT_ROOT / "error_detection_metrics.csv"),
        "pooled_oof_roc_figure": str(OUTPUT_ROOT / "pooled_oof_roc_curves.png"),
        "hybrid_fold_auc_figure": str(OUTPUT_ROOT / "hybrid_fold_auc.png"),
    },
    "post_test_rule": (
        "Do not change fold selection, hyperparameters, seeds, or model architecture "
        "in response to these outer-test results."
    ),
}

with open(
    OUTPUT_ROOT / "final_test_evaluation_manifest.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        final_evaluation_manifest,
        handle,
        indent=2,
    )

print("=" * 78)
print("FINAL OUTER-TEST EVALUATION COMPLETE")
print("=" * 78)
print(json.dumps(final_evaluation_manifest["hybrid_pooled_oof"], indent=2))
print("\nResults saved under:")
print(OUTPUT_ROOT)
